In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import csv
import time
import os
import re
from datetime import datetime
import logging

class FatSecretIndonesiaScraper:
    def __init__(self, output_dir="fatsecret_data", delay=1.5):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
            'Accept-Language': 'id-ID,id;q=0.9,en;q=0.8',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        })
        
        self.base_url = 'https://www.fatsecret.co.id'
        self.output_dir = output_dir
        self.delay = delay
        os.makedirs(output_dir, exist_ok=True)
        
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(f'{output_dir}/scraper.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
        
        self.recipes_data = []
        self.failed_urls = []
        
        # NEW: Dictionary to track categories for each recipe URL
        self.recipe_categories = {}

    # Fungsi normalisasi angka dengan satuan
    def _normalize_number_with_unit(self, value, unit=""):
        if value is None:
            return ""
        
        text = str(value).strip()
        num_match = re.search(r'(\d+(?:[.,]\d+)?)', text)
        if not num_match:
            return text  # kalau ga ada angka, return apa adanya
        
        num_str = num_match.group(1).replace(",", ".")
        try:
            num = float(num_str)
            num_formatted = str(num).replace(".", ",")
            return f"{num_formatted}{unit}"
        except:
            return text

    def discover_recipe_pages(self):
        """Temukan semua halaman kategori resep berdasarkan struktur yang diberikan"""
        recipe_pages = []
        
        # URL utama
        main_urls = [
            f'{self.base_url}/resep/',
            f'{self.base_url}/Default.aspx?pa=recsh'
        ]
        
        for url in main_urls:
            recipe_pages.append(('Halaman Utama Resep', url, 'Umum'))
        
        # Kategori berdasarkan struktur yang diberikan - UPDATED dengan kategori yang lebih spesifik
        categories = {
            'Populer': [
                ('Tinggi Nilai', '/resep/populer/tinggi-nilai/', 'Populer - Tinggi Nilai'),
                ('Terbaru', '/resep/populer/terbaru/', 'Populer - Terbaru'),
                ('Populer', '/resep/populer/', 'Populer')
            ],
            'Jenis Makanan': [
                ('Makanan Pembuka', '/resep/kategori/makanan-pembuka/', 'Makanan Pembuka'),
                ('Sup', '/resep/kategori/sup/', 'Sup'),
                ('Hidangan Utama', '/resep/kategori/hidangan-utama/', 'Hidangan Utama'),
                ('Lauk Pauk', '/resep/kategori/lauk-pauk/', 'Lauk Pauk'),
                ('Roti & Produk Panggang', '/resep/kategori/roti-produk-panggang/', 'Roti & Produk Panggang'),
                ('Salad dan Salad Dressing', '/resep/kategori/salad/', 'Salad dan Salad Dressing'),
                ('Saus dan Bumbu', '/resep/kategori/saus-bumbu/', 'Saus dan Bumbu'),
                ('Makanan Penutup', '/resep/kategori/makanan-penutup/', 'Makanan Penutup'),
                ('Makanan Ringan', '/resep/kategori/makanan-ringan/', 'Makanan Ringan'),
                ('Minuman', '/resep/kategori/minuman/', 'Minuman'),
                ('Lainnya', '/resep/kategori/lainnya/', 'Lainnya'),
                ('Makan Pagi', '/resep/kategori/makan-pagi/', 'Makan Pagi'),
                ('Makan Siang', '/resep/kategori/makan-siang/', 'Makan Siang'),
                ('Makan Malam', '/resep/kategori/makan-malam/', 'Makan Malam')
            ]
        }
        
        # Tambahkan semua kategori
        for category_type, sub_categories in categories.items():
            for sub_name, sub_url, category_name in sub_categories:
                full_url = urljoin(self.base_url, sub_url)
                recipe_pages.append((f"{category_type}: {sub_name}", full_url, category_name))
        
        # Coba scrape halaman utama untuk menemukan kategori tambahan
        try:
            response = self.session.get(f'{self.base_url}/resep/')
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Cari link kategori di sidebar "Temukan Resep Lainnya"
                sidebar = soup.find('td', class_='rightCell')
                if sidebar:
                    # Cari semua link yang mungkin merupakan kategori
                    possible_category_links = sidebar.find_all('a', href=re.compile(r'/resep/'))
                    for link in possible_category_links:
                        href = link.get('href')
                        text = link.get_text().strip()
                        if (href and text and 
                            '/resep/' in href and 
                            not re.search(r'/resep/\d+-', href) and  # bukan link resep individual
                            len(text) > 2 and
                            not any(x in text.lower() for x in ['selengkapnya', 'lihat', 'more', 'detail'])):
                            
                            full_url = urljoin(self.base_url, href)
                            # Cek apakah URL sudah ada
                            if not any(full_url == page[1] for page in recipe_pages):
                                category_name = f"Sidebar - {text}"
                                recipe_pages.append((f"Sidebar: {text}", full_url, category_name))
                
                # Cari link "Lihat Semua" atau pagination
                view_all_links = soup.find_all('a', href=True, text=re.compile(r'(Lihat Semua|View All|Selengkapnya)', re.IGNORECASE))
                for link in view_all_links:
                    href = link.get('href')
                    if href and '/resep/' in href:
                        full_url = urljoin(self.base_url, href)
                        if not any(full_url == page[1] for page in recipe_pages):
                            recipe_pages.append(('Lihat Semua Resep', full_url, 'Lihat Semua'))
                            
        except Exception as e:
            self.logger.warning(f"Error discovering additional pages: {e}")
        
        # Log semua halaman yang akan di-scrape
        self.logger.info(f"Ditemukan {len(recipe_pages)} halaman untuk di-scrape:")
        for i, (name, url, category) in enumerate(recipe_pages[:10], 1):  # Tampilkan 10 pertama
            self.logger.info(f"   {i}. {name} -> {url} [Kategori: {category}]")
        if len(recipe_pages) > 10:
            self.logger.info(f"   ... dan {len(recipe_pages) - 10} halaman lainnya")
        
        return recipe_pages

    def get_recipe_links_from_page(self, page_url, category_name):
        """Dapatkan link resep dari halaman kategori dengan handling sorting dan tracking kategori"""
        recipe_links = []
        try:
            self.logger.info(f"Scraping recipe links from: {page_url} [Kategori: {category_name}]")
            response = self.session.get(page_url)
            if response.status_code != 200:
                self.logger.warning(f"Cannot access {page_url} - Status: {response.status_code}")
                return recipe_links
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Method 1: Cari link resep dengan pattern yang spesifik
            pattern = re.compile(r'/resep/\d+-[^/]+/Default\.aspx')
            links = soup.find_all('a', href=pattern)
            
            for link in links:
                href = link.get('href')
                text = link.get_text().strip()
                if href and text:
                    full_url = urljoin(self.base_url, href)
                    recipe_links.append((full_url, text))
                    # NEW: Track kategori untuk setiap resep
                    self.recipe_categories[full_url] = category_name
            
            # Method 2: Cari di tabel atau container resep
            if not recipe_links:
                # Cari di tabel dengan class generic (biasanya berisi list resep)
                tables = soup.find_all('table', class_='generic')
                for table in tables:
                    links = table.find_all('a', href=pattern)
                    for link in links:
                        href = link.get('href')
                        text = link.get_text().strip()
                        if href and text:
                            full_url = urljoin(self.base_url, href)
                            recipe_links.append((full_url, text))
                            # NEW: Track kategori untuk setiap resep
                            self.recipe_categories[full_url] = category_name
            
            # Method 3: Cari di elemen dengan class borderBottom (biasanya item resep)
            if not recipe_links:
                recipe_items = soup.find_all(class_='borderBottom')
                for item in recipe_items:
                    links = item.find_all('a', href=pattern)
                    for link in links:
                        href = link.get('href')
                        text = link.get_text().strip()
                        if href and text:
                            full_url = urljoin(self.base_url, href)
                            recipe_links.append((full_url, text))
                            # NEW: Track kategori untuk setiap resep
                            self.recipe_categories[full_url] = category_name
            
            # Method 4: Cari semua link dengan pattern /resep/ID-judul/
            if not recipe_links:
                all_links = soup.find_all('a', href=True)
                for link in all_links:
                    href = link.get('href')
                    if href and re.search(r'/resep/\d+-[^/]+/Default\.aspx', href):
                        text = link.get_text().strip()
                        if text:
                            full_url = urljoin(self.base_url, href)
                            recipe_links.append((full_url, text))
                            # NEW: Track kategori untuk setiap resep
                            self.recipe_categories[full_url] = category_name
            
            # Hapus duplikat
            recipe_links = list(dict.fromkeys(recipe_links))
            self.logger.info(f"Found {len(recipe_links)} recipe links from {page_url} [Kategori: {category_name}]")
            
            # Jika tidak ditemukan link, coba dengan parameter sorting berbeda
            if not recipe_links and 'populer' in page_url.lower():
                self.logger.info(f"⚠️  Tidak ditemukan link di {page_url}, mencoba parameter sorting...")
                
                # Coba dengan parameter sorting yang berbeda
                sorting_options = [
                    '?sort=rating',  # sorting by rating
                    '?sort=date',    # sorting by date
                    '?sort=popular', # sorting by popularity
                    '?page=1',       # page 1
                    '?limit=50',     # limit 50
                ]
                
                for sort_param in sorting_options:
                    try:
                        sorted_url = f"{page_url.rstrip('/')}{sort_param}"
                        self.logger.info(f"   Mencoba dengan sorting: {sorted_url}")
                        
                        sorted_response = self.session.get(sorted_url)
                        if sorted_response.status_code == 200:
                            sorted_soup = BeautifulSoup(sorted_response.content, 'html.parser')
                            sorted_links = sorted_soup.find_all('a', href=pattern)
                            
                            for link in sorted_links:
                                href = link.get('href')
                                text = link.get_text().strip()
                                if href and text:
                                    full_url = urljoin(self.base_url, href)
                                    recipe_links.append((full_url, text))
                                    # NEW: Track kategori untuk setiap resep
                                    self.recipe_categories[full_url] = category_name
                            
                            if sorted_links:
                                self.logger.info(f"   ✅ Ditemukan {len(sorted_links)} link dengan sorting {sort_param}")
                                break
                            
                    except Exception as e:
                        self.logger.warning(f"   Error dengan sorting {sort_param}: {e}")
                        continue
            
        except Exception as e:
            self.logger.error(f"Error scraping recipe links from {page_url}: {e}")
        
        return recipe_links

    def get_all_recipe_links_with_pagination(self):
        """Dapatkan semua link resep dengan menangani pagination"""
        all_recipe_links = []
        recipe_pages = self.discover_recipe_pages()
        
        for page_name, page_url, category_name in recipe_pages:
            self.logger.info(f"📂 Scraping from: {page_name} - {page_url} [Kategori: {category_name}]")
            
            current_page = page_url
            page_count = 0
            max_pages = 20  # Safety limit
            
            while current_page and page_count < max_pages:
                try:
                    # Dapatkan link dari halaman saat ini
                    links = self.get_recipe_links_from_page(current_page, category_name)
                    all_recipe_links.extend(links)
                    
                    # Cari link ke halaman berikutnya
                    response = self.session.get(current_page)
                    if response.status_code == 200:
                        soup = BeautifulSoup(response.content, 'html.parser')
                        
                        next_page = None
                        
                        # Cari link pagination dengan berbagai pattern
                        next_selectors = [
                            'a.pagination-next', 'a.next', 'a[rel="next"]',
                            'a:contains("Next")', 'a:contains("Berikut")',
                            'a:contains(">")', 'a:contains("»")'
                        ]
                        
                        for selector in next_selectors:
                            try:
                                next_link = soup.select_one(selector)
                                if next_link:
                                    href = next_link.get('href')
                                    if href:
                                        next_page = urljoin(self.base_url, href)
                                        break
                            except:
                                continue
                        
                        # Jika tidak ditemukan, cari dengan text pattern
                        if not next_page:
                            for link in soup.find_all('a', href=True):
                                text = link.get_text().strip().lower()
                                if any(word in text for word in ['next', 'berikut', 'selanjutnya', '>', '»']):
                                    href = link.get('href')
                                    if href and not href.startswith('javascript:'):
                                        next_page = urljoin(self.base_url, href)
                                        break
                        
                        # Jika next_page sama dengan current_page, hentikan
                        if next_page == current_page or not next_page:
                            break
                        
                        current_page = next_page
                        page_count += 1
                        time.sleep(self.delay)
                        
                    else:
                        break
                        
                except Exception as e:
                    self.logger.error(f"Error processing page {current_page}: {e}")
                    break
        
        # Hapus duplikat
        unique_links = list(dict.fromkeys(all_recipe_links))
        self.logger.info(f"🎯 Total unique recipe links found: {len(unique_links)}")
        return unique_links

    def extract_nutrition_data_improved(self, soup, page_text):
        nutrition_data = {}
        page_text_lower = page_text.lower()
        
        try:
            # Kalori
            calorie_match = re.search(r'terdapat\s+(\d+(?:,\d+)?)\s*kalori\s+dalam', page_text_lower)
            if calorie_match:
                nutrition_data['kalori'] = self._normalize_number_with_unit(calorie_match.group(1), " kkal")
            
            kcal_match = re.search(r'(\d+(?:,\d+)?)\s+kkal', page_text)
            if kcal_match and 'kalori' not in nutrition_data:
                nutrition_data['kalori'] = self._normalize_number_with_unit(kcal_match.group(1), " kkal")
                
            energy_match = re.search(r'energi\s+\d+\s+kj\s+(\d+(?:,\d+)?)\s+kkal', page_text_lower)
            if energy_match and 'kalori' not in nutrition_data:
                nutrition_data['kalori'] = self._normalize_number_with_unit(energy_match.group(1), " kkal")
            
            # Nutrisi lain
            nutrients_patterns = {
                'lemak': (r'lemak\s+(\d+(?:,\d+)?)\s*g', " g"),
                'lemak_jenuh': (r'lemak jenuh\s+(\d+(?:,\d+)?)\s*g', " g"),
                'protein': (r'protein\s+(\d+(?:,\d+)?)\s*g', " g"),
                'karbohidrat': (r'karbohidrat\s+(\d+(?:,\d+)?)\s*g', " g"),
                'serat': (r'serat\s+(\d+(?:,\d+)?)\s*g', " g"),
                'gula': (r'gula\s+(\d+(?:,\d+)?)\s*g', " g"),
                'sodium': (r'sodium\s+(\d+(?:,\d+)?)\s*mg', " mg"),
                'kalium': (r'kalium\s+(\d+(?:,\d+)?)\s*mg', " mg"),
                'kolesterol': (r'kolesterol\s+(\d+(?:,\d+)?)\s*mg', " mg")
            }
            for nutrient, (pattern, unit) in nutrients_patterns.items():
                match = re.search(pattern, page_text_lower)
                if match:
                    nutrition_data[nutrient] = self._normalize_number_with_unit(match.group(1), unit)
            
        except Exception as e:
            self.logger.error(f"Error extracting nutrition with regex: {e}")
        
        # Parsing tabel HTML
        try:
            tables = soup.find_all('table')
            for table in tables:
                rows = table.find_all('tr')
                for row in rows:
                    cells = row.find_all(['td', 'th'])
                    if len(cells) >= 2:
                        label = cells[0].get_text().strip().lower()
                        value_text = cells[1].get_text().strip()
                        numeric_match = re.search(r'(\d+(?:[,\.]\d+)?)', value_text.replace(',', '.'))
                        if numeric_match:
                            if 'kalori' in label or 'energy' in label or 'kkal' in label:
                                if 'kalori' not in nutrition_data:
                                    nutrition_data['kalori'] = self._normalize_number_with_unit(numeric_match.group(1), " kkal")
                            elif 'lemak' in label and 'jenuh' not in label:
                                if 'lemak' not in nutrition_data:
                                    nutrition_data['lemak'] = self._normalize_number_with_unit(numeric_match.group(1), " g")
                            elif 'protein' in label:
                                if 'protein' not in nutrition_data:
                                    nutrition_data['protein'] = self._normalize_number_with_unit(numeric_match.group(1), " g")
                            elif 'karbohidrat' in label:
                                if 'karbohidrat' not in nutrition_data:
                                    nutrition_data['karbohidrat'] = self._normalize_number_with_unit(numeric_match.group(1), " g")
                            elif 'serat' in label:
                                if 'serat' not in nutrition_data:
                                    nutrition_data['serat'] = self._normalize_number_with_unit(numeric_match.group(1), " g")
                            elif 'gula' in label:
                                if 'gula' not in nutrition_data:
                                    nutrition_data['gula'] = self._normalize_number_with_unit(numeric_match.group(1), " g")
                            elif 'sodium' in label or 'natrium' in label:
                                if 'sodium' not in nutrition_data:
                                    nutrition_data['sodium'] = self._normalize_number_with_unit(numeric_match.group(1), " mg")
                            elif 'kalium' in label:
                                if 'kalium' not in nutrition_data:
                                    nutrition_data['kalium'] = self._normalize_number_with_unit(numeric_match.group(1), " mg")
        except Exception as e:
            self.logger.error(f"Error extracting nutrition from tables: {e}")
        
        return nutrition_data

    
    def extract_recipe_data(self, recipe_url, recipe_title):
        """Extract data lengkap dari halaman resep individual"""
        
        try:
            response = self.session.get(recipe_url)
            if response.status_code != 200:
                self.logger.warning(f"Cannot access recipe: {recipe_url}")
                return None
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            recipe_data = {
                'url': recipe_url,
                'title': recipe_title,
                'scraped_at': datetime.now().isoformat(),
                # NEW: Add category information
                'category': self.recipe_categories.get(recipe_url, 'Unknown')
            }
            
            # 1. Extract judul resep dari halaman
            title_selectors = ['h1', 'h2', '.recipe-title', '.recipesummary h1']
            for selector in title_selectors:
                title_elem = soup.select_one(selector)
                if title_elem and title_elem.get_text().strip():
                    recipe_data['page_title'] = title_elem.get_text().strip()
                    break
            
            # 2. Extract bahan-bahan
            ingredients = []
            ingredient_selectors = [
                '.ingredient', '.ingredients li', '.recipe-ingredients li',
                '.bahan li', '.bahan-bahan li'
            ]
            
            for selector in ingredient_selectors:
                elements = soup.select(selector)
                if elements:
                    ingredients = [elem.get_text().strip() for elem in elements 
                                 if elem.get_text().strip()]
                    if ingredients:
                        break
            
            recipe_data['ingredients'] = ingredients
            
            # 3. Extract cara masak/instruksi
            instructions = []
            instruction_selectors = [
                '.instruction', '.instructions li', '.recipe-instructions li',
                '.directions li', '.method li', '.cara li', '.langkah li'
            ]
            
            for selector in instruction_selectors:
                elements = soup.select(selector)
                if elements:
                    instructions = [elem.get_text().strip() for elem in elements 
                                  if elem.get_text().strip()]
                    if instructions:
                        break
            
            recipe_data['instructions'] = instructions
            
            # 4. Extract informasi nutrisi - MENGGUNAKAN METHOD YANG DIPERBAIKI
            page_text = soup.get_text()
            nutrition_data = self.extract_nutrition_data_improved(soup, page_text)
            
            # 5. Extract structured data (JSON-LD) - tetap gunakan sebagai backup
            structured_recipes = []
            scripts = soup.find_all('script', type='application/ld+json')
            
            for script in scripts:
                try:
                    if script.string:
                        data = json.loads(script.string)
                        
                        # Handle both single recipe and list of recipes
                        recipes_to_process = []
                        if isinstance(data, list):
                            recipes_to_process = data
                        else:
                            recipes_to_process = [data]
                        
                        for item in recipes_to_process:
                            if isinstance(item, dict) and item.get('@type') == 'Recipe':
                                structured_recipes.append(item)
                                
                except json.JSONDecodeError:
                    continue
            
            # Process structured data untuk nutrition - sebagai fallback
            if structured_recipes:
                main_recipe = structured_recipes[0]
                recipe_data['structured_data'] = main_recipe
                
                # Extract nutrition dari structured data
                if 'nutrition' in main_recipe and not nutrition_data:  # biar gak overwrite hasil parser manual
                    nutrition = main_recipe['nutrition']
                    if isinstance(nutrition, dict):
                        # Map nutrition values
                        nutrition_mapping = {
                            'calories': 'kalori',
                            'proteinContent': 'protein',
                            'carbohydrateContent': 'karbohidrat', 
                            'fatContent': 'lemak',
                            'saturatedFatContent': 'lemak_jenuh',
                            'fiberContent': 'serat',
                            'sugarContent': 'gula',
                            'sodiumContent': 'sodium'
                        }
                        
                        for eng_key, indo_key in nutrition_mapping.items():
                            if eng_key in nutrition:
                                nutrition_data[indo_key] = nutrition[eng_key]
                
                # Extract additional recipe info from structured data
                additional_fields = [
                    'description', 'prepTime', 'cookTime', 'totalTime',
                    'recipeYield', 'recipeCategory', 'recipeCuisine'
                ]
                for field in additional_fields:
                    if field in main_recipe:
                        recipe_data[field] = main_recipe[field]
            
            recipe_data['nutrition'] = nutrition_data
            
            # 6. Extract metadata tambahan
            # Servings/porsi
            servings_elem = soup.find(string=re.compile(r"Hasil:", re.IGNORECASE))
            if servings_elem:
                # cek next_sibling atau strong/span setelahnya
                sibling = servings_elem.find_next(string=True)
                hasil_text = ""
                if sibling:
                    hasil_text = sibling.strip()
                else:
                    # fallback: ambil parent tapi buang "Hasil:"
                    parent = servings_elem.find_parent()
                    if parent:
                        hasil_text = parent.get_text(strip=True).replace("Hasil:", "").strip()

                if hasil_text:
                    match = re.search(r'(\d+)', hasil_text)
                    if match:
                        recipe_data['recipeYield'] = match.group(1) + " porsi"
                    else:
                        recipe_data['recipeYield'] = hasil_text

            return recipe_data
        
        except Exception as e:
            self.logger.error(f"Error extracting recipe data from {recipe_url}: {e}")
            return None
    
    def save_data(self):
        """Save data ke berbagai format file"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # 1. Save to JSON (format lengkap)
        json_file = f'{self.output_dir}/recipes_{timestamp}.json'
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(self.recipes_data, f, ensure_ascii=False, indent=2)
        
        # 2. Save to CSV (format ringkas) - DIPERBAIKI untuk nutrition dan CATEGORY
        csv_file = f'{self.output_dir}/recipes_{timestamp}.csv'
        with open(csv_file, 'w', newline='', encoding='utf-8') as f:
            if self.recipes_data:
                fieldnames = ['title', 'url', 'page_title', 'category', 'ingredients', 
                            'instructions', 'kalori', 'protein', 'karbohidrat', 
                            'lemak', 'serat', 'gula', 'sodium', 'kalium', 'kolesterol',
                            'lemak_jenuh', 'description', 'prepTime', 'cookTime', 'recipeYield']
                
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                
                for recipe in self.recipes_data:
                    row = {
                        'title': recipe.get('title', ''),
                        'url': recipe.get('url', ''),
                        'page_title': recipe.get('page_title', ''),
                        'category': recipe.get('category', ''),  # NEW: Add category column
                        'ingredients': ' | '.join(recipe.get('ingredients', [])),
                        'instructions': ' | '.join(recipe.get('instructions', [])),
                        'description': recipe.get('description', ''),
                        'prepTime': recipe.get('prepTime', ''),
                        'cookTime': recipe.get('cookTime', ''),
                        'recipeYield': recipe.get('recipeYield', ''),
                    }
                    
                    # Add nutrition data - DIPERBAIKI
                    nutrition = recipe.get('nutrition', {})
                    nutrition_keys = ['kalori', 'protein', 'karbohidrat', 'lemak', 'serat', 
                                     'gula', 'sodium', 'kalium', 'kolesterol', 'lemak_jenuh']
                    
                    for key in nutrition_keys:
                        row[key] = nutrition.get(key, '')
                    
                    writer.writerow(row)
        
        # 3. Save detailed ingredients and instructions with CATEGORY
        detailed_file = f'{self.output_dir}/recipes_detailed_{timestamp}.json'
        detailed_data = []
        
        for recipe in self.recipes_data:
            detailed_recipe = {
                'title': recipe.get('page_title') or recipe.get('title'),
                'url': recipe.get('url'),
                'category': recipe.get('category', ''),  # NEW: Add category
                'ingredients': recipe.get('ingredients', []),
                'instructions': recipe.get('instructions', []),
                'nutrition': recipe.get('nutrition', {}),
                'metadata': {
                    'description': recipe.get('description', ''),
                    'prepTime': recipe.get('prepTime', ''),
                    'cookTime': recipe.get('cookTime', ''),
                    'totalTime': recipe.get('totalTime', ''),
                    'recipeYield': recipe.get('recipeYield', ''),
                    'category': recipe.get('recipeCategory', ''),
                    'cuisine': recipe.get('recipeCuisine', '')
                }
            }
            detailed_data.append(detailed_recipe)
        
        with open(detailed_file, 'w', encoding='utf-8') as f:
            json.dump(detailed_data, f, ensure_ascii=False, indent=2)
        
        # NEW: Save category statistics
        category_stats_file = f'{self.output_dir}/category_statistics_{timestamp}.json'
        category_stats = {}
        for recipe in self.recipes_data:
            category = recipe.get('category', 'Unknown')
            if category not in category_stats:
                category_stats[category] = {
                    'count': 0,
                    'recipes': []
                }
            category_stats[category]['count'] += 1
            category_stats[category]['recipes'].append({
                'title': recipe.get('page_title') or recipe.get('title'),
                'url': recipe.get('url')
            })
        
        with open(category_stats_file, 'w', encoding='utf-8') as f:
            json.dump(category_stats, f, ensure_ascii=False, indent=2)
        
        self.logger.info(f"Data saved to:")
        self.logger.info(f"  - JSON: {json_file}")
        self.logger.info(f"  - CSV: {csv_file}")
        self.logger.info(f"  - Detailed: {detailed_file}")
        self.logger.info(f"  - Category Stats: {category_stats_file}")
        
        # Save failed URLs
        if self.failed_urls:
            failed_file = f'{self.output_dir}/failed_urls_{timestamp}.txt'
            with open(failed_file, 'w', encoding='utf-8') as f:
                for url in self.failed_urls:
                    f.write(f"{url}\n")
            self.logger.info(f"  - Failed URLs: {failed_file}")
    
    def run_complete_scraping(self, max_recipes=None):
        """Jalankan proses scraping lengkap dengan metode yang diperbaiki"""
        
        self.logger.info("🇮🇩 MEMULAI SCRAPING FATSECRET INDONESIA WITH CATEGORY TRACKING")
        self.logger.info("="*60)
        
        start_time = time.time()
        
        # 1. Discover semua halaman resep dengan metode baru
        self.logger.info("\n🔍 MENCARI HALAMAN RESEP DAN KATEGORI...")
        recipe_pages = self.discover_recipe_pages()
        
        # 2. Kumpulkan semua link resep dengan pagination handling
        self.logger.info("\n📋 MENGUMPULKAN LINK RESEP DARI SEMUA HALAMAN...")
        total_recipe_links = self.get_all_recipe_links_with_pagination()
        
        # 3. Coba juga dapatkan link dari sitemap (sebagai backup)
        self.logger.info("\n🗺️  MENCARI LINK DARI SITEMAP...")
        sitemap_links = self.get_links_from_sitemap()
        if sitemap_links:
            total_recipe_links.extend(sitemap_links)
            self.logger.info(f"   Ditemukan {len(sitemap_links)} link dari sitemap")
        
        # Hapus duplikasi
        unique_links = list(dict.fromkeys(total_recipe_links))
        self.logger.info(f"\n📊 TOTAL UNIQUE RECIPE LINKS: {len(unique_links)}")
        
        # NEW: Show category distribution
        category_distribution = {}
        for url, title in unique_links:
            category = self.recipe_categories.get(url, 'Unknown')
            category_distribution[category] = category_distribution.get(category, 0) + 1
        
        self.logger.info(f"\n📊 DISTRIBUSI KATEGORI:")
        for category, count in sorted(category_distribution.items(), key=lambda x: x[1], reverse=True):
            self.logger.info(f"   {category}: {count} resep")
        
        # Tampilkan beberapa contoh link untuk debugging
        if unique_links:
            self.logger.info("\n🔗 CONTOH LINK YANG DITEMUKAN:")
            for i, (url, title) in enumerate(unique_links[:5], 1):
                category = self.recipe_categories.get(url, 'Unknown')
                self.logger.info(f"   {i}. [{category}] {title[:50]}... -> {url}")
            if len(unique_links) > 5:
                self.logger.info(f"   ... dan {len(unique_links) - 5} link lainnya")
        
        # Limit jika diminta
        if max_recipes:
            unique_links = unique_links[:max_recipes]
            self.logger.info(f"   ⚠️  LIMITED TO: {max_recipes} recipes")
        
        # 4. Scrape setiap resep
        self.logger.info(f"\n🍳 MULAI SCRAPING {len(unique_links)} RESEP")
        self.logger.info("-"*60)
        
        successful_scrapes = 0
        failed_scrapes = 0
        
        for i, (recipe_url, recipe_title) in enumerate(unique_links, 1):
            category = self.recipe_categories.get(recipe_url, 'Unknown')
            self.logger.info(f"[{i}/{len(unique_links)}] [{category}] {recipe_title[:30]}...")
            
            try:
                recipe_data = self.extract_recipe_data(recipe_url, recipe_title)
                
                if recipe_data:
                    self.recipes_data.append(recipe_data)
                    ingredients_count = len(recipe_data.get('ingredients', []))
                    instructions_count = len(recipe_data.get('instructions', []))
                    nutrition_count = len(recipe_data.get('nutrition', {}))
                    
                    # Log data nutrisi untuk debugging
                    nutrition_summary = ""
                    if recipe_data.get('nutrition'):
                        nutrition = recipe_data['nutrition']
                        if 'kalori' in nutrition:
                            nutrition_summary += f" Kal:{nutrition['kalori']}"
                        if 'protein' in nutrition:
                            nutrition_summary += f" Prot:{nutrition['protein']}"
                        if 'lemak' in nutrition:
                            nutrition_summary += f" Lemak:{nutrition['lemak']}"
                    
                    self.logger.info(f"   ✅ SUCCESS - Bahan: {ingredients_count}, "
                                f"Langkah: {instructions_count}, Nutrisi: {nutrition_count}"
                                f"{nutrition_summary}")
                    successful_scrapes += 1
                else:
                    self.failed_urls.append(recipe_url)
                    self.logger.warning(f"   ❌ FAILED - Data tidak berhasil diekstrak")
                    failed_scrapes += 1
                    
            except Exception as e:
                self.failed_urls.append(recipe_url)
                self.logger.error(f"   💥 ERROR - {str(e)}")
                failed_scrapes += 1
            
            # Jeda antar request
            time.sleep(self.delay)
            
            # Progress update setiap 10 resep
            if i % 10 == 0:
                success_rate = (successful_scrapes / i) * 100
                self.logger.info(f"   📊 PROGRESS: {i}/{len(unique_links)} - "
                            f"Success: {successful_scrapes} ({success_rate:.1f}%)")
        
        # 5. Save results
        if self.recipes_data:
            self.save_data()
        
        # 6. Final statistics
        end_time = time.time()
        duration = end_time - start_time
        
        self.logger.info(f"\n📈 SCRAPING SELESAI")
        self.logger.info("="*60)
        self.logger.info(f"⏰ Total waktu: {duration:.1f} detik ({duration/60:.1f} menit)")
        self.logger.info(f"✅ Resep berhasil: {len(self.recipes_data)}")
        self.logger.info(f"❌ Resep gagal: {len(self.failed_urls)}")
        
        if unique_links:
            success_rate = (len(self.recipes_data) / len(unique_links)) * 100
            self.logger.info(f"📊 Success rate: {success_rate:.1f}%")
        
        # Statistics per resep
        if self.recipes_data:
            avg_ingredients = sum(len(r.get('ingredients', [])) for r in self.recipes_data) / len(self.recipes_data)
            avg_instructions = sum(len(r.get('instructions', [])) for r in self.recipes_data) / len(self.recipes_data)
            recipes_with_nutrition = sum(1 for r in self.recipes_data if r.get('nutrition'))
            
            # NEW: Category statistics untuk hasil scraping
            final_category_stats = {}
            for recipe in self.recipes_data:
                category = recipe.get('category', 'Unknown')
                final_category_stats[category] = final_category_stats.get(category, 0) + 1
            
            # Statistik nutrisi
            nutrition_stats = {}
            for recipe in self.recipes_data:
                nutrition = recipe.get('nutrition', {})
                for nutrient in nutrition.keys():
                    nutrition_stats[nutrient] = nutrition_stats.get(nutrient, 0) + 1
            
            self.logger.info(f"\n📊 STATISTIK RESEP:")
            self.logger.info(f"   Rata-rata bahan per resep: {avg_ingredients:.1f}")
            self.logger.info(f"   Rata-rata langkah per resep: {avg_instructions:.1f}")
            self.logger.info(f"   Resep dengan info nutrisi: {recipes_with_nutrition} ({recipes_with_nutrition/len(self.recipes_data)*100:.1f}%)")
            
            self.logger.info(f"\n📊 HASIL KATEGORI YANG BERHASIL DI-SCRAPE:")
            for category, count in sorted(final_category_stats.items(), key=lambda x: x[1], reverse=True):
                percentage = (count / len(self.recipes_data)) * 100
                self.logger.info(f"   {category}: {count} resep ({percentage:.1f}%)")
            
            if nutrition_stats:
                self.logger.info(f"\n🥗 NUTRISI YANG BERHASIL DIEKSTRAK:")
                for nutrient, count in sorted(nutrition_stats.items()):
                    percentage = (count / len(self.recipes_data)) * 100
                    self.logger.info(f"   {nutrient}: {count} resep ({percentage:.1f}%)")
        
        # Tampilkan beberapa contoh resep yang berhasil
        if self.recipes_data:
            self.logger.info(f"\n🎉 CONTOH RESEP YANG BERHASIL:")
            for i, recipe in enumerate(self.recipes_data[:3], 1):
                title = recipe.get('page_title', recipe.get('title', 'Unknown'))
                category = recipe.get('category', 'Unknown')
                ingredients = len(recipe.get('ingredients', []))
                instructions = len(recipe.get('instructions', []))
                nutrition = len(recipe.get('nutrition', {}))
                self.logger.info(f"   {i}. [{category}] {title[:30]}... - "
                            f"Bahan: {ingredients}, Langkah: {instructions}, Nutrisi: {nutrition}")
        
        # Simpan URL yang gagal untuk analisis
        if self.failed_urls:
            failed_file = f'{self.output_dir}/failed_urls_detailed_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
            with open(failed_file, 'w', encoding='utf-8') as f:
                for url in self.failed_urls:
                    f.write(f"{url}\n")
            self.logger.info(f"\n📝 Daftar URL gagal disimpan di: {failed_file}")
        
        return self.recipes_data

    def get_links_from_sitemap(self):
        """Coba dapatkan link resep dari sitemap"""
        sitemap_urls = [
            f'{self.base_url}/sitemap.xml',
            f'{self.base_url}/sitemap_recipes.xml',
            f'{self.base_url}/sitemap/resep.xml'
        ]
        
        recipe_links = []
        for sitemap_url in sitemap_urls:
            try:
                response = self.session.get(sitemap_url)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'xml')
                    urls = soup.find_all('loc')
                    for url in urls:
                        url_text = url.get_text()
                        if '/resep/' in url_text and re.search(r'/resep/\d+-', url_text):
                            # Extract title from URL
                            title_match = re.search(r'/resep/\d+-(.+)', url_text)
                            title = title_match.group(1).replace('-', ' ').title() if title_match else "Unknown"
                            recipe_links.append((url_text, title))
                            # NEW: Set default category untuk sitemap links
                            self.recipe_categories[url_text] = 'Sitemap'
            except:
                continue
        
        return recipe_links

# Contoh penggunaan - UPDATED dengan category tracking
if __name__ == "__main__":
    # Inisialisasi scraper
    scraper = FatSecretIndonesiaScraper(
        output_dir="fatsecret_indonesia_data_with_categories",
        delay=2  # 2 detik jeda antar request
    )
    
    # Jalankan scraping semua resep
    results = scraper.run_complete_scraping()
    
    print(f"\n🎉 SCRAPING SELESAI! Berhasil mengumpulkan {len(results)} resep Indonesia")
    print(f"📁 Data tersimpan di folder: {scraper.output_dir}")
    
    # NEW: Tampilkan ringkasan kategori
    if results:
        category_summary = {}
        for recipe in results:
            category = recipe.get('category', 'Unknown')
            category_summary[category] = category_summary.get(category, 0) + 1
        
        print(f"\n📊 RINGKASAN KATEGORI:")
        for category, count in sorted(category_summary.items(), key=lambda x: x[1], reverse=True):
            print(f"   {category}: {count} resep")
        
        # Tampilkan contoh data nutrisi per kategori
        recipes_with_nutrition = [r for r in results if r.get('nutrition')]
        print(f"\n🧬 Contoh resep dengan data nutrisi ({len(recipes_with_nutrition)} dari {len(results)} total):")
        
        categories_shown = set()
        for recipe in recipes_with_nutrition[:5]:
            category = recipe.get('category', 'Unknown')
            if category not in categories_shown:
                print(f"\nKategori: {category}")
                print(f"Resep: {recipe.get('page_title', recipe.get('title'))}")
                nutrition = recipe.get('nutrition', {})
                for nutrient, value in list(nutrition.items())[:3]:  # Show first 3 nutrients
                    print(f"  {nutrient}: {value}")
                categories_shown.add(category)
                if len(categories_shown) >= 3:  # Limit to 3 categories for brevity
                    break

2025-09-05 15:08:16,864 - INFO - 🇮🇩 MEMULAI SCRAPING FATSECRET INDONESIA WITH CATEGORY TRACKING
2025-09-05 15:08:16,866 - INFO - ============================================================
2025-09-05 15:08:16,867 - INFO - 
🔍 MENCARI HALAMAN RESEP DAN KATEGORI...
C:\Users\sisil\AppData\Local\Temp\ipykernel_14264\1561255919.py:131: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  view_all_links = soup.find_all('a', href=True, text=re.compile(r'(Lihat Semua|View All|Selengkapnya)', re.IGNORECASE))
2025-09-05 15:08:17,931 - INFO - 🎯 Ditemukan 32 halaman untuk di-scrape:
2025-09-05 15:08:17,932 - INFO -    1. Halaman Utama Resep -> https://www.fatsecret.co.id/resep/ [Kategori: Umum]
2025-09-05 15:08:17,933 - INFO -    2. Halaman Utama Resep -> https://www.fatsecret.co.id/Default.aspx?pa=recsh [Kategori: Umum]
2025-09-05 15:08:17,936 - INFO -    3. Populer: Tinggi Nilai -> https://www.fatsecret.co.id/resep/populer/tinggi-nilai/ [Kategori:


🎉 SCRAPING SELESAI! Berhasil mengumpulkan 315 resep Indonesia
📁 Data tersimpan di folder: fatsecret_indonesia_data_with_categories

📊 RINGKASAN KATEGORI:
   Sidebar - Makan Siang: 98 resep
   Sidebar - Makan Pagi: 71 resep
   Sidebar - Lauk Pauk: 38 resep
   Sidebar - Hidangan Utama: 34 resep
   Sidebar - Makanan Ringan: 20 resep
   Sidebar - Roti & Produk Panggang: 11 resep
   Sidebar - Lainnya: 10 resep
   Sidebar - Sup: 10 resep
   Sidebar - Salad dan Salad Dressing: 7 resep
   Sidebar - Minuman: 6 resep
   Sidebar - Saus dan Bumbu: 4 resep
   Sidebar - Makanan Penutup: 4 resep
   Sidebar - Makanan Pembuka: 2 resep

🧬 Contoh resep dengan data nutrisi (315 dari 315 total):

Kategori: Sidebar - Makan Pagi
Resep: Bolu Pisang Coklat
  kalori: 136,0 kkal
  lemak: 4,62 g
  lemak_jenuh: 1,677 g

Kategori: Sidebar - Makan Siang
Resep: Telur Dadar Sayur
  kalori: 268,0 kkal
  lemak: 16,58 g
  lemak_jenuh: 9,935 g
